# Module 10: Model Versioning

**Snowpark ML Fundamentals — Week 3 | Lunch & Learn**

---

## Learning Objectives
- Train multiple model variants and register them as versions
- Browse and compare versions side-by-side
- Set the default version for inference
- Manage metrics post-registration

## Key Concept
> **Versioning** lets you iterate on models without losing previous work.
> Each version is an immutable snapshot with its own metrics, aliases, and
> callable functions — all governed by Snowflake RBAC.

---

In [1]:
%load_ext autoreload
%autoreload 2

## 10.1 Setup

Create session, generate a churn dataset, preprocess, and split.

In [2]:
import sys
sys.path.insert(0, '../src')

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

from snowpark_fundamentals.session import create_session
from snowpark_fundamentals.data.sample_data import create_customer_churn_dataset
from snowpark_fundamentals.data.loader import split_data
from snowpark_fundamentals.preprocessing.transformers import build_preprocessing_pipeline
from snowpark_fundamentals.modeling.trainer import train_model, predict
from snowpark_fundamentals.modeling.evaluation import evaluate_binary_classifier

session = create_session(env_path='../.env')

NUMERIC_COLS = ['AGE', 'TENURE_MONTHS', 'MONTHLY_CHARGES', 'TOTAL_CHARGES',
                'SUPPORT_TICKETS', 'USAGE_HOURS_PER_WEEK', 'NUM_DEPENDENTS']
CATEGORICAL_COLS = ['PLAN_TYPE', 'CONTRACT_TYPE', 'PAYMENT_METHOD']
LABEL_COL = 'CHURNED'
FEATURE_COLS = [f"{c}_SCALED" for c in NUMERIC_COLS] + [f"{c}_ENCODED" for c in CATEGORICAL_COLS]

df = create_customer_churn_dataset(session, n_rows=5000)
df_processed, _ = build_preprocessing_pipeline(df, NUMERIC_COLS, CATEGORICAL_COLS)
train_df, test_df = split_data(df_processed)
print(f"Train: {train_df.count()} rows, Test: {test_df.count()} rows")

Train: 3962 rows, Test: 1038 rows


In [3]:
train_df.show(5)

-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"PLAN_TYPE_ENCODED"  |"CONTRACT_TYPE_ENCODED"  |"PAYMENT_METHOD_ENCODED"  |"CUSTOMER_ID"  |"AGE"  |"PLAN_TYPE"  |"TENURE_MONTHS"  |"MONTHLY_CHARGES"  |"SUPPORT_TICKETS"  |"USAGE_HOURS_PER_WEEK"  |"CONTRACT_TYPE"  |"PAYMENT_METHOD"  |"NUM_DEPENDENTS"  |"TOTAL_CHARGES"  |"CHURNED"  |"AGE_SCALED"   |"TENURE_MONTHS_SCALED"  |"MONTHLY_CHARGES_SCALED"  |"TOTAL_CHARGES_SCALED"  |"SUPPORT_TICKETS_SCALED"  |"USAGE_HOURS_PER_WEEK_SCALED"  |"NUM_DEPENDENTS_SCALED"  |
----------------------------------------------------------------------------

In [4]:
test_df.show(5)

-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"PLAN_TYPE_ENCODED"  |"CONTRACT_TYPE_ENCODED"  |"PAYMENT_METHOD_ENCODED"  |"CUSTOMER_ID"  |"AGE"  |"PLAN_TYPE"  |"TENURE_MONTHS"  |"MONTHLY_CHARGES"  |"SUPPORT_TICKETS"  |"USAGE_HOURS_PER_WEEK"  |"CONTRACT_TYPE"  |"PAYMENT_METHOD"  |"NUM_DEPENDENTS"  |"TOTAL_CHARGES"  |"CHURNED"  |"AGE_SCALED"   |"TENURE_MONTHS_SCALED"  |"MONTHLY_CHARGES_SCALED"  |"TOTAL_CHARGES_SCALED"  |"SUPPORT_TICKETS_SCALED"  |"USAGE_HOURS_PER_WEEK_SCALED"  |"NUM_DEPENDENTS_SCALED"  |
----------------------------------------------------------------------------

## 10.2 Train Two Variants

**V1**: XGBoost with default hyperparameters.  
**V2**: Tuned — more trees, deeper, lower learning rate.

This is the most common versioning pattern: same algorithm, different hyperparameters.

In [5]:
# V1: XGBoost defaults
model_v1 = train_model(
    train_df, FEATURE_COLS, LABEL_COL,
    model_type='xgboost',
    model_params={'n_estimators': 100, 'max_depth': 6},
)
preds_v1 = predict(model_v1, test_df)
metrics_v1 = evaluate_binary_classifier(preds_v1, LABEL_COL, 'PREDICTION')
print("V1 metrics:", metrics_v1)

The version of package 'snowflake-snowpark-python' in the local environment is 1.45.0, which does not fit the criteria for the requirement 'snowflake-snowpark-python'. Your UDF might not work when the package version is different between the server and your local environment.
The version of package 'snowflake-telemetry-python' in the local environment is 0.7.1, which does not fit the criteria for the requirement 'snowflake-telemetry-python'. Your UDF might not work when the package version is different between the server and your local environment.
DataFrame.flatten() is deprecated since 0.7.0. Use `DataFrame.join_table_function()` instead.


V1 metrics: {'accuracy': 0.819846, 'precision': 0.7102803738317757, 'recall': 0.3275862068965517, 'f1_score': 0.44837758112094395}


In [6]:
# V2: Tuned hyperparameters
model_v2 = train_model(
    train_df, FEATURE_COLS, LABEL_COL,
    model_type='xgboost',
    model_params={'n_estimators': 200, 'max_depth': 8, 'learning_rate': 0.05},
)
preds_v2 = predict(model_v2, test_df)
metrics_v2 = evaluate_binary_classifier(preds_v2, LABEL_COL, 'PREDICTION')
print("V2 metrics:", metrics_v2)

The version of package 'snowflake-snowpark-python' in the local environment is 1.45.0, which does not fit the criteria for the requirement 'snowflake-snowpark-python'. Your UDF might not work when the package version is different between the server and your local environment.
The version of package 'snowflake-telemetry-python' in the local environment is 0.7.1, which does not fit the criteria for the requirement 'snowflake-telemetry-python'. Your UDF might not work when the package version is different between the server and your local environment.


V2 metrics: {'accuracy': 0.842004, 'precision': 0.925, 'recall': 0.31896551724137934, 'f1_score': 0.4743589743589744}


## 10.3 Register Both Versions

Each call to `log_model` creates an **immutable version**. The first version
automatically becomes the DEFAULT.

In [7]:
from snowpark_fundamentals.registry.model_registry import get_registry, log_model, delete_model

current_db = session.get_current_database().replace('"', '')
current_schema = session.get_current_schema().replace('"', '')
registry = get_registry(session, current_db, current_schema)

# Clean up from previous runs (idempotent)
try:
    delete_model(registry, 'CHURN_PREDICTOR_W3')
    print("Deleted existing CHURN_PREDICTOR_W3 (re-run cleanup)")
except Exception:
    pass  # Model doesn't exist yet — first run

sample_input = test_df.select(FEATURE_COLS).limit(10)

# Register V1
mv1 = log_model(
    registry=registry,
    model=model_v1.to_xgboost(),
    model_name='CHURN_PREDICTOR_W3',
    version_name='V1',
    sample_input=sample_input,
    metrics=metrics_v1,
)
print("Registered CHURN_PREDICTOR_W3 V1")

Deleted existing CHURN_PREDICTOR_W3 (re-run cleanup)
Model logged successfully.: 100%|██████████| 6/6 [00:21<00:00,  3.63s/it]                          
Registered CHURN_PREDICTOR_W3 V1


In [8]:
# Register V2
mv2 = log_model(
    registry=registry,
    model=model_v2.to_xgboost(),
    model_name='CHURN_PREDICTOR_W3',
    version_name='V2',
    sample_input=sample_input,
    metrics=metrics_v2,
)
print("Registered CHURN_PREDICTOR_W3 V2")

Model logged successfully.: 100%|██████████| 6/6 [00:17<00:00,  2.97s/it]                          
Registered CHURN_PREDICTOR_W3 V2


## 10.4 Browse Versions

Every model tracks three automatic aliases:
- **DEFAULT** — used when no version is specified
- **FIRST** — the earliest version
- **LAST** — the most recently registered version


Key behaviors:                                                                                                                                           
- FIRST and LAST are automatic — you can't move them manually                                                                                              
- DEFAULT starts on V1, then moves only when you call set_default_version()                                                                                
- Custom aliases (production, staging, archived) are manually assigned via set_model_alias() ... we are gonna see this in the next file.
- A version with no aliases shows []                                                                                                                       
- One version can have multiple aliases simultaneously    

In [9]:
from snowpark_fundamentals.registry.model_registry import list_versions

versions_df = list_versions(registry, 'CHURN_PREDICTOR_W3')
print(versions_df[['name', 'aliases', 'is_default_version']].to_string())

  name              aliases is_default_version
0   V1  ["DEFAULT","FIRST"]               true
1   V2             ["LAST"]              false


## 10.5 Compare Metrics

Use `compare_model_versions()` to pull metrics from multiple versions
into a single view. This is essential for deciding which version to promote.

In [10]:
import pandas as pd
from snowpark_fundamentals.registry.model_registry import compare_model_versions

comparison = compare_model_versions(registry, 'CHURN_PREDICTOR_W3', ['V1', 'V2'])

# Flatten into a pandas DataFrame for side-by-side comparison
rows = []
for item in comparison:
    row = {'version': item['version']}
    row.update(item['metrics'])
    rows.append(row)

comparison_df = pd.DataFrame(rows).set_index('version')
print(comparison_df.to_string())

         accuracy  precision    recall  f1_score
version                                         
V1       0.819846    0.71028  0.327586  0.448378
V2       0.842004    0.92500  0.318966  0.474359


## 10.6 Set Default Version

The DEFAULT alias determines which version is used when callers
don't specify a version. Promote V2 to default if its metrics are better.

In [11]:
from snowpark_fundamentals.registry.model_registry import set_default_version

set_default_version(registry, 'CHURN_PREDICTOR_W3', 'V2')
print("Default version set to V2")

# Verify
versions_df = list_versions(registry, 'CHURN_PREDICTOR_W3')
print(versions_df[['name', 'aliases', 'is_default_version']].to_string())

Default version set to V2
  name             aliases is_default_version
0   V1           ["FIRST"]              false
1   V2  ["DEFAULT","LAST"]               true


## 10.7 Manage Metrics

Metrics can be added or updated **after** registration. This is useful for:
- Adding evaluation metrics computed on a holdout set
- Tracking data drift scores over time
- Storing business KPIs alongside model metrics

In [12]:
from snowpark_fundamentals.registry.model_registry import (
    get_model_metrics,
    set_model_metrics,
)

# View current metrics for V2
current_metrics = get_model_metrics(registry, 'CHURN_PREDICTOR_W3', 'V2')
print("Current V2 metrics:", current_metrics)

# Add a new metric (e.g., data drift score from a monitoring pipeline)
set_model_metrics(registry, 'CHURN_PREDICTOR_W3', 'V2', {'data_drift': 0.02})

# Verify
updated_metrics = get_model_metrics(registry, 'CHURN_PREDICTOR_W3', 'V2')
print("Updated V2 metrics:", updated_metrics)

Current V2 metrics: {'accuracy': 0.842004, 'precision': 0.925, 'recall': 0.31896551724137934, 'f1_score': 0.4743589743589744}
Updated V2 metrics: {'accuracy': 0.842004, 'precision': 0.925, 'recall': 0.31896551724137934, 'f1_score': 0.4743589743589744, 'data_drift': 0.02}


## Key Takeaways

1. **Versions are immutable** — once registered, a version's model artifact cannot change
2. **Metrics are mutable** — you can add, update, or delete metrics at any time
3. **DEFAULT alias** controls which version is used in unversioned calls
4. **FIRST/LAST aliases** are automatic — they track the oldest and newest versions
5. `compare_model_versions()` enables **data-driven promotion decisions**

---

**Next: [11 — Model Lifecycle](11_model_lifecycle.ipynb)** — Alias-based deployment, promotion workflows, and metadata

In [13]:
session.close()